## 1. Load it, and remove the empty columns
Load *bike_rentals.csv* and make a copy. Print the shape and the data types. Which type does *price* have? Then find the columns where every value is missing, print their names, and remove them. How many columns are left, and how many empty columns?

In [1]:
import pandas as pd

rentals_raw = pd.read_csv("data/bike_rentals.csv")
rentals = rentals_raw.copy()

# 403 rows and 15 columns
print(rentals.shape)

# price has str
print(rentals.dtypes)

# coupon_code, damage_notes, late_fee_note are all empty columns
print(rentals.columns[rentals.isna().all()])

rentals = rentals.dropna(axis=1, how="all")

# 403 rows and 12 columns
print(rentals.shape)

# rental now has no empty columns
print(rentals.columns[rentals.isna().all()])

(403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object
Index(['coupon_code', 'damage_notes', 'late_fee_note'], dtype='str')
(403, 12)
Index([], dtype='str')


## 2. The price: from text to numbers
Look at the first rows of *price*. It is text with a currency in front and a comma for thousands. remove the 2 things in 2 separate steps, and print the first 3 values after each step. Then change the type to float, check the type, and run `describe()` on the price. Finally, show the 3 most expensive rentals `(rental_id, bike_type, status, price)`.

In [2]:
print(rentals["price"].head(3))

# remove currency prefix
rentals["price"] = rentals["price"].str.replace("SEK", "", regex=False).str.strip()
print(rentals["price"].head(3))

# remove comma
rentals["price"] =  rentals["price"].str.replace(",", "", regex=False)
print(rentals["price"].head(3))

# convert into float
rentals["price"] = rentals["price"].astype(float)
print(rentals["price"].head(3))

# type is float64
print(rentals["price"].dtype)

# highest price is 3401 and lowest price is 154.5
print(rentals["price"].describe())

print(rentals[["rental_id", "bike_type", "status", "price"]].
      sort_values("price", ascending=False).
      head(3))

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
Name: price, dtype: str
0      855.50
1    1,311.50
2      350.00
Name: price, dtype: str
0     855.50
1    1311.50
2     350.00
Name: price, dtype: str
0     855.5
1    1311.5
2     350.0
Name: price, dtype: float64
float64
count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64
    rental_id   bike_type     status   price
187    R-5282  Cargo bike  Completed  3401.0
49     R-5209  Cargo bike  Completed  2979.5
270    R-5378      E-bike  Completed  2941.0


## 3. The missing prices
How many prices are missing, and how many percent is that? Is there a reason? Compare with the column *status*: use `pd.crosstab` on *"price is missing"* and *"the rental was cancelled"*. Which missing prices have a reason, and which are missing by accident? Write what you decide to do with them, as a comment.

In [3]:
# 51 prices are missing, which is 12.7%
print(rentals["price"].isna().sum())
print(((rentals["price"].isna().mean()) * 100).round(1))

# 47 missing prices are because they were cancelled, and 4 are by accident
# I would leave them as missing since filling would invent fake prices
print(pd.crosstab(rentals["price"].isna(),
                  rentals["status"] == "Cancelled"))

51
12.7
status  False  True 
price               
False     352      0
True        4     47


## 4. The letters t and f
Find the columns that contain only *t* and *f* (and maybe missing values), with a list comprehension. Why is *status* not in the list? Change the columns to `True`and `False` with `map`. How many missing values does each column have afterwards? Then compare the median price of members and non-members, with the number of rentals behind each number.

In [4]:
# status is not in the result since it has different values than "t" or "f"
tf_columns = [col for col in rentals.columns if set(rentals[col].dropna().unique()) <= {"t", "f"}]
print(tf_columns)

for col in tf_columns:
    rentals[col] = rentals[col].map({"t": True, "f": False})

# there are only 3 rentals where the value is missing
# and theyre all in the insured column
print(rentals[tf_columns].isna().sum())

# members pay less (558 instead of 756)
rentals.groupby("is_member")["price"].agg(["count", "median"]).round(0)

['is_member', 'insured', 'paid_online']
is_member      0
insured        3
paid_online    0
dtype: int64


,count,median
is_member,,
False,248,756.0
True,104,558.0


## 5. The dates
*start_date* and *end_date* are text. Count the missing values in both columns before, change them to dates with `errors="coerce"`, and count again. Did something change? Check the types. Then create the column *rental_days* (the end minus the start, in days) and run `describe()` on it. Something looks wrong at the minimum. Write what you see as a comment. You do not need to fix it in this part.

In [5]:
# start_date has 0 missing values, end_date has 0
print(rentals["start_date"].isna().sum())
print(rentals["end_date"].isna().sum())

rentals["start_date"] = pd.to_datetime(rentals["start_date"], errors="coerce")
rentals["end_date"] = pd.to_datetime(rentals["end_date"], errors="coerce")

# start_date has 2 missing values, end_date has 0
print(rentals["start_date"].isna().sum())
print(rentals["end_date"].isna().sum())

# both are datetime64[us]
print(rentals["start_date"].dtype)
print(rentals["end_date"].dtype)

rentals["rental_days"] = (rentals["end_date"] - rentals["start_date"]).dt.days

# min is -2, which means some rentals end before they start
print(rentals["rental_days"].describe())

0
0
2
0
datetime64[us]
datetime64[us]
count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64


## 6. The missing ratings
How many ratings are missing? Compare with *status* with `pd.crosstab`. In the lecture the missing ratings were exactly the listings without reviews. Is it a perfect match this time? Create a colmun *has_rating* and count it. Run `describe()`on *rating* and write down the median.

In [6]:
# ratings has 115 missing values
print(rentals["rating"].isna().sum())

# 47 cancelled, 46 completed,22 no-show
# it's not a perfect match
print(pd.crosstab(rentals["rating"].isna(), rentals["status"]))

rentals["has_rating"] = rentals["rating"].notna()

# 288 are True, 115 are False
print(rentals["has_rating"].value_counts())

# median is 5
print(rentals["rating"].describe())

115
status  Cancelled  Completed  No-show
rating                               
False           0        288        0
True           47         46       22
has_rating
True     288
False    115
Name: count, dtype: int64
count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64


## 7. The helmet text
Look at *helmet_text* with `value_counts(dropna=False)`. Create two new columns: *child_helmet* (True if the text contains the word child) and *helmets_count* (the number in the text). The text *"No helmets"* has no number. What does the number get for those rows, and how do you make it 0 instead? Count the values in helmet_count.

In [7]:
print(rentals["helmets_text"].value_counts(dropna=False))

rentals["child_helmet"] = rentals["helmets_text"].str.contains("child")

no_helmet = rentals["helmets_text"].str.lower().str.contains("no")
number = rentals["helmets_text"].str.extract(r"([\d.]+)", expand=False).astype(float)
rentals["helmets_count"] = number.where(~no_helmet, 0)

# 1: 193
# 2: 127
# 3: 20
# 0: 59
# NaN : 4
print(rentals["helmets_count"].value_counts(dropna=False))

# for rentals that explicitly say theyre not renting helmets like, "no helmets" i would change it to a 0
# for rentals with no text at all, i would let both number and text stay NaN as we have no idea how many helmets they rented


helmets_text
1 helmet          146
2 helmets         127
No helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64
helmets_count
1.0    193
2.0    127
0.0     59
3.0     20
NaN      4
Name: count, dtype: int64
